# Dioptra datasets → Kaggle upload (one dataset per env / scene)

Uploads each dataset folder from Google Drive to Kaggle as its own **public** dataset under `yumnamharryson/`.
Covers TartanGround (5 envs) + Tartanair indoor complement (5 envs) + Hypersim whole pack (1) + NYU-v2 (1) = 12 datasets.
Run top-to-bottom in Colab. Uploads resume safely — re-running a cell skips finished datasets.

## 0. Config — edit if needed

In [ ]:
KAGGLE_USERNAME = "yumnamharryson"
DIO = "/content/drive/MyDrive/dioptra_datasets"
# slug -> drive folder (slugs must be lowercase, alphanumeric + dashes)
ENVS = {
    "tartanground-office-stereo-depth": f"{DIO}/tartanground/Office",
    "tartanground-hospital-stereo-depth": f"{DIO}/tartanground/Hospital",
    "tartanground-constructionsite-stereo-depth": f"{DIO}/tartanground/ConstructionSite",
    "tartanground-oldindustrialcity-stereo-depth": f"{DIO}/tartanground/OldIndustrialCity",
    "tartanground-modular-neighborhood-stereo-depth": f"{DIO}/tartanground/ModularNeighborhoodIntExt",
    "tartanair-indoors-abandonedschool": f"{DIO}/tartanair_indoor_complement/AbandonedSchool",
    "tartanair-indoors-restaurant": f"{DIO}/tartanair_indoor_complement/Restaurant",
    "tartanair-indoors-hospital": f"{DIO}/tartanair_indoor_complement/hospital",
    "tartanair-indoors-office": f"{DIO}/tartanair_indoor_complement/office",
    "tartanair-indoors-office2": f"{DIO}/tartanair_indoor_complement/office2",
    "hypersim-pack": f"{DIO}/hypersim",  # whole package, one dataset
    "nyu-v2-depth": f"{DIO}/nyu_v2",
}
STAGE_DIR = "/tmp/kaggle_upload"  # per-env staging (symlinks, no copy)

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Verify dataset is visible

In [ ]:
import os
missing = [p for p in ENVS.values() if not os.path.isdir(p)]
assert not missing, f"Missing Drive folders: {missing}"
print(f"All {len(ENVS)} env folders visible.")
!du -sh "/content/drive/MyDrive/dioptra_datasets/tartanground" "/content/drive/MyDrive/dioptra_datasets/tartanair_indoor_complement"
!find "/content/drive/MyDrive/dioptra_datasets/tartanground" "/content/drive/MyDrive/dioptra_datasets/tartanair_indoor_complement" -name '*.zip' | wc -l
!find "/content/drive/MyDrive/dioptra_datasets/tartanground" "/content/drive/MyDrive/dioptra_datasets/tartanair_indoor_complement" -name '*.part' | head  # expect empty (no partials)

## 3. Install + authenticate Kaggle API

Get `kaggle.json` from Kaggle → Account → API → *Create New Token*, then upload it to Colab when prompted (or place it in Drive and copy it).

In [ ]:
!pip install -q -U kaggle  # need >=1.8 for KGAT_ tokens
!kaggle -v

In [ ]:
import os, shutil
# Option A (new-style token): paste your KGAT_... token when prompted.
# It lives only in this Colab session's memory/env — never written to the notebook.
import getpass
tok = os.environ.get("KAGGLE_API_TOKEN", "")
if not tok:
    tok = getpass.getpass("Paste KGAT_ token (or Enter to use kaggle.json instead): ").strip()
if tok:
    os.environ["KAGGLE_API_TOKEN"] = tok
    os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
    open(os.path.expanduser("~/.kaggle/access_token"), "w").write(tok)
    print("KGAT token configured for this session.")
else:
    # Option B: upload kaggle.json (legacy) via Colab file picker
    from google.colab import files
    UPLOADED = list(files.upload().keys()) if not os.path.exists(os.path.expanduser("~/.kaggle/kaggle.json")) else []
    if UPLOADED:
        os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
        shutil.move(UPLOADED[0], os.path.expanduser("~/.kaggle/kaggle.json"))
        os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
        print("kaggle.json installed from upload.")
    # Option C: copy from Drive (uncomment + set path if you stored it there)
    # !mkdir -p ~/.kaggle && cp "/content/drive/MyDrive/kaggle.json" ~/.kaggle/kaggle.json && chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets list --mine 2>&1 | head -n 20  # auth check

## 4. Upload — one Kaggle dataset per env (public)

Each env is staged via symlink (no 150 GB copy) and uploaded with `kaggle datasets create` (first time) or `version` (updates). Re-run safely resumes.

In [ ]:
import json, os, subprocess

def run(cmd):
    print("+", " ".join(cmd))
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(r.stdout[-3000:])
    if r.returncode != 0:
        print(r.stderr[-3000:])
    return r

os.makedirs(STAGE_DIR, exist_ok=True)
print(f"Total Kaggle datasets to upload: {len(ENVS)}")
for slug, src in ENVS.items():
    assert os.path.exists(src), f"Missing Drive path: {src}"
    env = os.path.basename(src.rstrip("/"))
    stage = os.path.join(STAGE_DIR, slug)
    os.makedirs(stage, exist_ok=True)
    # Mirror the tree with per-FILE symlinks (kaggle's zip follows file links;
    # a single dir symlink would be skipped by os.walk). .zip scenes are
    # linked as-is - no need to unzip, Kaggle stores them directly.
    if os.path.isdir(src):
        n = 0
        for root, dirs, files in os.walk(src):
            rel = os.path.relpath(root, src)
            dest_root = stage if rel == "." else os.path.join(stage, rel)
            os.makedirs(dest_root, exist_ok=True)
            for fn in files:
                link = os.path.join(dest_root, fn)
                if not os.path.lexists(link):
                    os.symlink(os.path.join(root, fn), link)
                    n += 1
        print(f"  staged {n} new file links")
    else:
        link = os.path.join(stage, env)
        if not os.path.lexists(link):
            os.symlink(src, link)
        print(f"  staged single file: {env}")
    meta = {
        "title": slug.replace("-", " ").title(),
        "id": f"{KAGGLE_USERNAME}/{slug}",
        "licenses": [{"name": "CC0-1.0"}],
    }
    with open(os.path.join(stage, "dataset-metadata.json"), "w") as f:
        json.dump(meta, f, indent=2)
    print(f"\n===== {env} -> {meta['id']} =====")
    # create if new, else new version (--dir-mode zip uploads folders)
    r = run(["kaggle", "datasets", "create", "-p", stage, "--dir-mode", "zip", "--public"])
    if r.returncode != 0 and ("already in use" in (r.stdout + r.stderr) or "already exists" in (r.stdout + r.stderr)):
        run(["kaggle", "datasets", "version", "-p", stage, "-m", "update", "--dir-mode", "zip"])

## 5. Verify uploads

In [ ]:
!kaggle datasets list --mine 2>&1 | head -n 20